# Feature Transform Engine (FTE) in Vertex AI Tabular Workflows

This notebook demonstrates how to configure and utilize the **Feature Transform Engine (FTE)** for automated and explicit feature engineering in Google Cloud Vertex AI Tabular Workflows.

## Overview: Automated vs. Explicit Feature Engineering

In Vertex AI Tabular Workflows, feature engineering transforms raw input data into numerical representations suitable for machine learning models:
- **Automated Feature Engineering (`"auto"`)**: By default, Vertex AI automatically infers data types and applies appropriate transformations (e.g., standard scaling for numbers, one-hot encoding for categories) based on dataset statistics.
- **Explicit Feature Engineering**: For greater control, custom business logic, or optimization, developers can explicitly specify column-level transformation strategies using a JSON configuration file.

### Supported Column Transform Types
1. `"categorical"`: Generates categorical encodings such as one-hot encodings, target encodings, or dictionary lookups.
2. `"numeric"`: Applies numerical scaling and normalization (e.g., standard z-score scaling, min-max normalization).
3. `"timestamp"`: Extracts datetime components (e.g., year, month, day, hour, day of week, time since epoch).
4. `"text"`: Processes free-form text using text embeddings or bag-of-words representations.
5. `"auto"`: Enables automated inferencing where Vertex AI selects the best transformation strategy for the column.

In [ ]:
import json

from dotenv import load_dotenv
from google.cloud import storage

from tabflows import (
    TabularPipelineConfig,
    create_tabular_pipeline_job,
    generate_fte_transformations,
    write_fte_transformations,
)

# Load environment variables from local .env file
load_dotenv()
print("Environment and libraries loaded successfully.")

In [ ]:
# Define explicit FTE column transformations dictionary for dataset columns
fte_column_types = {
    "age": "numeric",
    "balance": "numeric",
    "day": "numeric",
    "duration": "numeric",
    "campaign": "numeric",
    "pdays": "numeric",
    "previous": "numeric",
    "job": "categorical",
    "marital": "categorical",
    "education": "categorical",
    "default": "categorical",
    "housing": "categorical",
    "loan": "categorical",
    "contact": "categorical",
    "month": "categorical",
    "poutcome": "categorical",
    "signup_date": "timestamp",
    "customer_notes": "text",
}

print(f"Defined explicit FTE column transform specifications for {len(fte_column_types)} columns.")

In [ ]:
# Generate FTE transformation specs dictionary list
fte_transformations = generate_fte_transformations(fte_column_types)

# Display formatted JSON specification
print("Generated FTE Transformation Specification JSON:")
print(json.dumps(fte_transformations, indent=2))

In [ ]:
# Initialize configuration and GCS storage client
config = TabularPipelineConfig()
storage_client = storage.Client(project=config.project_id)

# Specify GCS URI destination for transform configuration
gcs_transform_config_uri = (
    f"{config.bucket_uri}/automl_tabular_pipeline/custom_fte_transform_config.json"
)

# Write FTE transformations JSON spec to Cloud Storage
write_fte_transformations(storage_client, gcs_transform_config_uri, fte_column_types)
print(f"Successfully uploaded FTE transform config to Cloud Storage:\n{gcs_transform_config_uri}")

In [ ]:
# Initialize TabularPipelineConfig with custom transform config path
fte_config = TabularPipelineConfig(
    transform_config_path=gcs_transform_config_uri,
)

print(
    "Initialized TabularPipelineConfig with FTE transform config path:\n"
    f"{fte_config.transform_config_path}"
)

# Create the pipeline job with custom FTE configuration
job_id = "fte-automl-tabular-full-search"
job = create_tabular_pipeline_job(fte_config, job_id=job_id)
print(f"PipelineJob object '{job_id}' created successfully with custom FTE transformations.")